# Nemotron 3.5 Lightning on Colab (vLLM) for the AML investigators

Serves `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4` (21.6 GB, 4-bit weights) with vLLM's
OpenAI-compatible API. The AML **investigators** use it instead of NVIDIA's rate-limited hosted API;
the senior investigator, tier agent and rule-update agent keep using Nemotron 3 Ultra on NVIDIA's API.

**Runtime:** Runtime → Change runtime type → **G4 GPU** (RTX PRO 6000 Blackwell, 96 GB; native 4-bit, fastest)
or **A100**. L4 and T4 are too small.

Run the cells in order. The last cell prints the lines to paste into `.env.local` on your Mac.
Keep this tab open and leave cell 7 running: it keeps the tunnel and session alive.

In [ ]:
# 1. Check the GPU
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu)
assert any(name in gpu for name in ("RTX PRO", "Blackwell", "A100", "H100", "L40")), "Choose the G4 (RTX PRO 6000) or A100 runtime: this model needs about 30 GB of GPU memory"

In [ ]:
# 2. Install vLLM (the model needs vLLM 0.27.1 or newer) and cloudflared for the tunnel
!pip install -q "vllm>=0.27.1"
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [ ]:
# 3. Start the server in the background (first start downloads ~22 GB and takes 5-10 minutes)
import secrets, subprocess, os
API_KEY = secrets.token_urlsafe(24)          # protects the public tunnel; paste it into .env.local
MODEL = "nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4"
log = open("/content/vllm.log", "w")
server = subprocess.Popen([
    "vllm", "serve", MODEL,
    "--served-model-name", "nemotron-3.5-lightning",
    "--api-key", API_KEY,
    "--host", "127.0.0.1", "--port", "8000",
    "--max-model-len", "32768",              # the agent's packets are below 24k tokens
    "--max-num-seqs", "32",                  # bounded Mamba cache; plenty for 3 parallel teams
    "--gpu-memory-utilization", "0.90",
    "--enable-prefix-caching",
    "--mamba-ssm-cache-dtype", "float16",
    "--reasoning-parser", "nemotron_v3",
], stdout=log, stderr=subprocess.STDOUT)
print("started; follow progress in the next cell")

In [ ]:
# 4. Wait until the model is loaded
import time, urllib.request
for _ in range(120):
    try:
        req = urllib.request.Request("http://127.0.0.1:8000/v1/models", headers={"Authorization": f"Bearer {API_KEY}"})
        print(urllib.request.urlopen(req, timeout=5).read()[:200]); break
    except Exception:
        time.sleep(10)
else:
    print(open("/content/vllm.log").read()[-3000:])

In [ ]:
# 5. Check structured JSON output (the agent relies on json_schema responses)
import json, urllib.request
body = {"model": "nemotron-3.5-lightning", "messages": [{"role": "user", "content": "Return ok"}], "max_tokens": 32,
        "temperature": 0, "chat_template_kwargs": {"enable_thinking": False},
        "response_format": {"type": "json_schema", "json_schema": {"name": "t", "strict": True, "schema": {
            "type": "object", "properties": {"ok": {"type": "string", "maxLength": 5}}, "required": ["ok"], "additionalProperties": False}}}}
req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions", data=json.dumps(body).encode(),
                             headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"})
print(json.loads(urllib.request.urlopen(req, timeout=120).read())["choices"][0]["message"]["content"])

In [ ]:
# 6. Open a tunnel and print the settings for the AML agent
import re, subprocess, time
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0); break
print("Paste into .env.local on your Mac, then restart the AML API:\n")
print(f"AML_INVESTIGATOR_BASE_URL={url}/v1")
print("AML_INVESTIGATOR_MODEL=nemotron-3.5-lightning")
print(f"AML_INVESTIGATOR_KEY={API_KEY}")
print("AML_INVESTIGATOR_RPM=240")

In [ ]:
# 7. Keep the tunnel and the session alive (leave this cell running)
import threading, time, urllib.request
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()   # drain cloudflared logs
while True:
    try:
        req = urllib.request.Request("http://127.0.0.1:8000/v1/models", headers={"Authorization": f"Bearer {API_KEY}"})
        urllib.request.urlopen(req, timeout=10); print(time.strftime("%H:%M"), "server ok", end="\r")
    except Exception as error:
        print(time.strftime("%H:%M"), "server check failed:", error)
    time.sleep(60)
